### Libraries

In [22]:
import os
import random
import json

import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad

import mygene
import pickle

In [23]:
from collections import Counter

In [24]:
from functions import *

In [ ]:
assert "OPENAI_API_KEY" in os.environ

In [ ]:
from openai import OpenAI

client = OpenAI()

### Setting the paths

In [27]:
data_dir = "/scratch/ahakobyan/single_cellm_data/"
project_dir = '/users/anna.hakobyan/projects/single-cellm/'

### Getting the annotations and samples

In [28]:
annotation_file = data_dir + "archs4_geo/annotations.json"
 
with open(annotation_file) as json_file:
    annots = json.load(json_file)

In [29]:
processed_annotation_file = data_dir + "archs4_geo/processed_annotations.json"
 
with open(processed_annotation_file) as json_file:
    processed_annots = json.load(json_file)

In [30]:
sample_types = [elem["sample_type"] for key, elem in annots.items()]

sample_type_counts = Counter(sample_types)

# sample_type_counts ===== 
# Counter({'cell_line': 234119,
#          'tissue': 173790,
#          'primary_cells': 148331,
#          'in_vitro_differentiated_cells': 55550,
#          'stem_cells': 34860,
#          'induced_pluripotent_stem_cells': 7038})

# treatment = [elem["treatment"] for key, elem in annots.items()]

sample_groups = {}
for type in sample_type_counts:
    sample_groups[type] = []

for sampleid, elem in annots.items():
    sample_groups[elem["sample_type"]].append(sampleid)

In [31]:
random.seed(50)

# selected_sample_ids = []
# selected_sample_ids.extend(sample_groups["induced_pluripotent_stem_cells"])
# selected_sample_ids.extend(random.sample(sample_groups["cell_line"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["tissue"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["primary_cells"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["in_vitro_differentiated_cells"], 28600 ) )
# selected_sample_ids.extend(random.sample(sample_groups["stem_cells"], 28600 ) )

# with open(data_dir + "selected_sample_ids.txt", 'w') as f:
#     for sid in selected_sample_ids:
#         f.write(sid + "\n")

with open(data_dir + "selected_sample_ids.txt", "r") as f:
    selected_sample_ids = [ elem.strip() for elem in f.readlines()]

In [ ]:
len(selected_sample_ids)

### Importing the gene_ranks

In [32]:
gene_ranks = sc.read_h5ad(data_dir + "archs4_geo/gene_ranks/gene_ranks_csr.h5ad")

### Getting the common sample IDs

In [33]:
common_samples = gene_ranks.obs.index.intersection(selected_sample_ids)

### Adding gene symbol to gene ranks anndata

In [21]:
# import mygene
# mg = mygene.MyGeneInfo()
# ginfo = mg.querymany(gene_ranks.var.index, scopes='ensembl.gene')


# gene_mapping = {}
# for gene in ginfo:
#     try:
#         gene_mapping[gene["query"]] = gene["symbol"]
#     except KeyError:
#         gene_mapping[gene["query"]] = gene["query"]

# gene_ranks.var["symbol"] = [gene_mapping[i] for i in gene_ranks.var.index]

# gene_ranks.write_h5ad(data_dir + "archs4_geo/gene_ranks/gene_ranks_csr.h5ad")

1 input query terms found dup hits:	[('ENSG00000268674', 3)]
11 input query terms found no hit:	['ENSG00000112096', 'ENSG00000168078', 'ENSG00000180525', 'ENSG00000189144', 'ENSG00000215271', 'ENS


### Get ranked genes for a sample 

In [45]:
sortout = np.argsort(gene_ranks["SRX185896"].layers["gene_ranks"])

In [53]:
sortout

ArrayView([[11975,  5513, 17852, ...,  7323,  8809, 12056]])

In [51]:
gene_ranks["SRX185896"].layers["gene_ranks"][0, sortout]

ArrayView([[    0,     1,     2, ..., 22309, 22310, 22311]], dtype=int32)

### Retrieving gene ranks as a function

In [31]:
random.choice(common_samples)

'SRX4576656'

In [30]:
sample_genes = get_sample_ranked_genes('SRX4576656', gene_ranks, N = 10)

### Check if GPT recognizes the sample from the gene list

#### Sample descriptions

sample 1: SRX186538
{'geo_title': 'CstF64-iCLIP rep3',
 'geo_metadata': 'strategy: iCLIP-Seq,cell line: HeLa,clip antibody: CstF64',
 'geo_source_name': 'HeLa',
 'sample_type': 'cell_line',
 'mapped_ontology_terms': 'HeLa, uterine cervix, disease, HeLa, cultured cell, female organism, cell line, African American, adenocarcinoma, adenocarcinoma, disease',
 'raw_biosample_metadata': 'source_name: HeLa; strategy: iCLIP-Seq; cell_line: HeLa; clip antibody: CstF64',
 'treatment': ''}

sample 2: SRX188848
{'geo_title': 'RNA_Seq_48hr_2',
 'geo_metadata': 'cell type: H1 embryonic stem cells,drug treatment: Activin,drug concentration: 50ng/ml,duration of treatment: 48hr',
 'geo_source_name': 'RNA-seq H1-hESC-48hr-Activin',
 'sample_type': 'stem_cells',
 'mapped_ontology_terms': 'embryonic stem cell, stem cell, treatment',
 'raw_biosample_metadata': 'source_name: RNA-seq H1-hESC-48hr-Activin; cell_type: H1 embryonic stem cells; drug treatment: Activin; drug concentration: 50ng/ml; duration of treatment: 48hr',
 'treatment': ''}

In [41]:
sample_1 = "SRX186538"

sample_2 = "SRX188848"
cmp = gpt_sample_guess_from_ranked_genes(sample_1, sample_2, gene_ranks,
                                   system_prompt,  gpt_model = "gpt-3.5-turbo", N = 100)

## match = True

In [42]:
cmp

{'sample1': 'SRX186538',
 'sample2': 'SRX188848',
 'choice': 1,
 'match': False,
 'reason': 'The sample is most associated with the genes: TP53TG3D, DSC3, CCND3, ANKRD35, RPL13, PADI6, CD19, DUSP22, FOXD4L3, NPHS1, NANOG, TNFAIP8L2, ZNF32.'}

In [ ]:
Nexp = 100

In [37]:
system_prompt = f"""
You are given an RNA-seq sample annotation and two gene lists. Your task is to say which gene list describes the sample best.

In the answer provide two lines. The first line tells which genes in the list were most associated with the sample. The second line contains only one number - "1" or "2" for gene list 1 or gene list 2.
Remember the second line contains one number that indicates your decision.
"""

# system_prompt = f"""
# You are given an RNA-seq sample annotation and two gene lists. Your task is to say which gene list describes the sample best.

# Give an explanation for your choice.
# """

In [91]:
gene_rank_test_outs = []
random.seed(10)
for i in range(Nexp):
    # print(i)
    sample_1 = random.choice(common_samples)
    sample_2 = random.choice(common_samples)
    out = gpt_sample_guess_from_ranked_genes(sample_1, sample_2, 
                                       gene_ranks, 
                                       system_prompt, 
                                       gpt_model = "gpt-3.5-turbo", N = 100)
    gene_rank_test_outs.append(out)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [67]:
sample_1

'SRX15686264'

In [69]:
annots[sample_1]

{'geo_title': 'Donor 3 - M0',
 'geo_metadata': 'cell type: Monocyte-derived macrophages,treatment: M0,donor: 3',
 'geo_source_name': 'Monocyte-derived macrophages',
 'sample_type': 'in_vitro_differentiated_cells',
 'mapped_ontology_terms': 'myeloid leukocyte, hematopoietic cell, macrophage, treatment, leukocyte',
 'raw_biosample_metadata': 'source_name: Monocyte-derived macrophages; cell_type: Monocyte-derived macrophages; treatment: M0; donor: 3',
 'treatment': 'treatment: M0'}

['ENSG00000288053', 'PRR33', 'ENSG00000288208', 'ENSG00000288258', 'SMIM42']

In [68]:
sample_2

'SRX1739148'

In [70]:
annots[sample_2]

{'geo_title': 'AF105_S2',
 'geo_metadata': 'individual: AF105,cell type: macrophage,infection: Salmonella,time: 2 hr',
 'geo_source_name': 'Macrophages',
 'sample_type': 'primary_cells',
 'mapped_ontology_terms': 'leukocyte, hematopoietic cell, myeloid leukocyte, macrophage',
 'raw_biosample_metadata': 'source_name: Macrophages; individual: AF105; cell_type: macrophage; infection: Salmonella; time: 2 hr',
 'treatment': ''}

In [80]:
random.seed(16)
a = gpt_sample_guess_from_ranked_genes("SRX1739148", "SRX15686264", 
                                       gene_ranks, 
                                       system_prompt, 
                                       gpt_model = "gpt-3.5-turbo", N = 100)

In [81]:
a

{'gpt_response': ChatCompletion(id='chatcmpl-8pd5NzzYKYT0dC5zmSWcZlPVL4fh1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The genes most associated with the sample are: ENSG00000286185, ENSG00000286192, ENSG00000286221, ENSG00000286231, ENSG00000286235\n2', role='assistant', function_call=None, tool_calls=None))], created=1707315665, model='gpt-3.5-turbo-0613', object='chat.completion', system_fingerprint=None, usage=CompletionUsage(completion_tokens=50, prompt_tokens=1247, total_tokens=1297)),
 'output': {'sample1': 'SRX1739148',
  'sample2': 'SRX15686264',
  'choice': 2,
  'match': True,
  'reason': 'The genes most associated with the sample are: ENSG00000286185, ENSG00000286192, ENSG00000286221, ENSG00000286231, ENSG00000286235'}}

In [92]:
Counter([elem["output"]["match"] for elem in gene_rank_test_outs])

Counter({True: 35, False: 34, None: 31})

### Testing with GPT4

In [93]:
gene_rank_test_outs_gpt4 = []
random.seed(10)
for i in range(Nexp):
    # print(i)
    sample_1 = random.choice(common_samples)
    sample_2 = random.choice(common_samples)
    out = gpt_sample_guess_from_ranked_genes(sample_1, sample_2, 
                                       gene_ranks, 
                                       system_prompt, 
                                       gpt_model = "gpt-4-1106-preview", N = 100)
    gene_rank_test_outs_gpt4.append(out)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


In [87]:
Counter([elem["output"]["match"] for elem in gene_rank_test_outs_gpt4])

Counter({False: 66, True: 34})

In [89]:
gene_rank_test_outs_gpt4[1]

{'gpt_response': ChatCompletion(id='chatcmpl-8pdXNgEIiyBiJMzszaWOYHyV3Oi5A', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='genes: NOTCH2NLC, BAD, HSCs are known to express NOTCH signaling pathway genes. BAD is known to be important in regulating apoptosis, which could be relevant to stem cell regulation. Neither gene lists are an ideal match, however, the first list includes NOTCH2NLC, which is related to the NOTCH signaling pathway known to be active in HSCs (hematopoietic stem cells). There are no genes specifically associated with HSC or AAVS1 KO (a common gene-editing target site) in the second list. \n1', role='assistant', function_call=None, tool_calls=None))], created=1707317401, model='gpt-4-1106-preview', object='chat.completion', system_fingerprint='fp_b4fb435f51', usage=CompletionUsage(completion_tokens=119, prompt_tokens=1217, total_tokens=1336)),
 'output': {'sample1': 'SRX10985668',
  'sample2': 'SRX1133927',
  'choic

### Testing with the deduplicated dataset

In [31]:
deduplicated_annotation_file = project_dir + "resources/human_disease/human_disease_strictly_deduplicated/cluster_centers.sample_ids_and_text.json"
 
with open(deduplicated_annotation_file) as json_file:
    dedup_annots = json.load(json_file)

In [32]:
dedup_annots 

{'DRX021334': "This sample is from a patient diagnosed with acute myeloid leukemia, a subtype of blood cancer. The sample was taken from the patient's blood, specifically from hematopoietic tissue. The patient has not undergone any treatment at the time of sampling. The genotype of the sample is NPM1 WT FLT3 WT. The study aims to identify a specific signature of long non-coding RNAs (LncRNAs) in acute myeloid leukemia with intermediate risk.",
 'DRX021335': "This sample is from a healthy individual's normal skin tissue. It is a primary sample and has not undergone any treatment. The sample was used in a study investigating long non-coding RNAs in psoriatic and healthy skin. The sample is labeled as 'GSM1930376: C12; Homo sapiens; RNA-Seq'.",
 'DRX021337': 'The sample is from a primary colorectal cancer tissue, specifically from the colon. It has not undergone any treatment. The sample is associated with diseases of cellular proliferation and neoplasms in the large intestine. The RNA-Se

In [35]:
set(dedup_annots.keys()).intersection({"DRX021340"})

{'DRX021340'}

### I still have to calculate gene ranks for these samples

### Trying out with individual samples with supposedly clear marker genes

In [37]:
# SRX375659 "\nThe sample is cord blood CD4+ cells, which are a type of immune cell. No treatments were applied to the sample.",
sid = "SRX375659"
teg = get_sample_ranked_genes(sid, gene_ranks, 100)
teg_str = ", ".join(teg["symbol"].tolist())
teg_str

# GPT4 it's unlikely that the entire set is specific to these cells without additional supporting data.

'COPS9, RPS6KA5, GAGE12B, H3C13, DNAJC5G, APEH, PPBP, HIC2, KRTAP4-5, MRPS6, IQCB1, ABCF1, ENSG00000268083, RNF182, GLT1D1, TENM4, SMIM15, NPY2R, GMNC, CMKLR1, RFLNB, COX3, MUC12, GAD2, ANXA7, ATP6V1G1, GNAT2, STX16-NPEPL1, MIR4713, ZBTB14, TUFT1, SOWAHA, CEP350, RBM25, MCU, ARHGEF39, COMMD6, PCDHAC2, MIR29A, CYB561D1, C6orf141, CSNKA2IP, NEMF, ARHGEF11, MICB, SLC52A2, DNAAF4, CLDN23, RDH14, CNEP1R1, CFAP43, MIR7976, C4B, HAPLN4, TYW1, NBPF6, SIPA1L3, RNLS, JRKL, GRIK5, BMAL1, TENM2, SLC5A2, ORMDL2, FAM91A1, PTGIR, CKLF, FIP1L1, DIS3, TMEM259, ZBTB9, PDS5B, MIR6741, TMEM181, LINC01588, EFNA4-EFNA3, NUFIP1, TECPR2, RUSC2, FAXC, SPP1, MIR4705, STK31, PPP2R3B, MYOCD, MIR548P, RNASE8, CCR7, KDELR2, MIR6751, ENSG00000286070, MIR4753, NAA11, BLK, CFD, SLC25A10, ZNF354C, CTDSPL, MIR2277, MIR548BB'

In [38]:
# "SRX1059099": "\nThe sample is a primary cell sample from a human donor, specifically pancreatic islet cells. No strain information is available."

sid = "SRX1059099"
teg = get_sample_ranked_genes(sid, gene_ranks, 100)
teg_str = ", ".join(teg["symbol"].tolist())
teg_str

# GPT4 
# NEUROG3: This is a critical transcription factor for the development of pancreatic islets.
# PC (Pyruvate Carboxylase): Important for glucose-stimulated insulin secretion in β-cells of the pancreas.

'ENSG00000258973, DUSP12, KRTAP9-6, ZNF135, RPP14, LGR6, FKRP, MIR5087, TBC1D3F, KRTAP1-3, NLRP11, CORT, ZNF311, PINX1, GRIA4, LOC128092248, ZNF860, CNTNAP3C, MIR128-2, IFI27L1, KLHL34, FOXD1, ANKRD20A3P, GKAP1, TXNL4A, PHF10, CCND2, TRIM56, ENSG00000285827, FANCA, OR52N2, OR4N4C, IKBKE-AS1, USHBP1, KRT31, LMLN, MIR563, MIR2467, MIR2054, ENSG00000249967, GEN1, FBXO17, PC, ENSG00000285526, TBC1D3I, LYSMD4, TBC1D3E, MIR3591, MIR3913-2, BRME1, PRH2, MIR4491, FASTKD5, CLN3, PRR19, RNF220, FGF5, UBA3, MIR584, MIR6730, RFXAP, VGF, VAMP8, RNASEK-C17orf49, MIR624, GSX1, TAS2R46, ARL1, NEUROG3, MIR623, LINC02875, DOLPP1, CCL15, CIMAP1D, CHD9NB, TRIM73, CCNT2, CXCR3, SRGAP3, C5orf22, SKI, KRTAP5-1, PRR23A, SNX18, AHCYL2, DDR1, NOXA1, TMEM234, C21orf58, ADRA2C, MIR329-1, AKIP1, SLC4A3, RPS6, LSM10, CTHRC1, KIRREL1, NPC1, ENSG00000228144, EID3'

In [ ]:
# 

### Getting the keywords from the model

In [14]:
with open(project_dir + "resources/archs4/smp_exp_keywords.pickle", 'rb') as picklefile:
    embed_keywords = pickle.load(picklefile)

In [14]:
embed_keywords.keys()

dict_keys(['SRX185895', 'SRX185896', 'SRX185897', 'SRX185898', 'SRX185899', 'SRX185900', 'SRX186158', 'SRX186159', 'SRX186160', 'SRX186161', 'SRX186162', 'SRX186163', 'SRX186164', 'SRX186165', 'SRX186166', 'SRX186167', 'SRX186168', 'SRX186169', 'SRX186170', 'SRX186171', 'SRX186536', 'SRX186537', 'SRX186538', 'SRX188847', 'SRX188848', 'SRX189728', 'SRX189729', 'SRX189730', 'SRX189731', 'SRX190849', 'SRX195293', 'SRX195294', 'SRX195295', 'SRX195296', 'SRX195297', 'SRX195298', 'SRX195299', 'SRX195300', 'SRX195301', 'SRX195461', 'SRX195462', 'SRX195463', 'SRX195464', 'SRX195465', 'SRX198056', 'SRX198069', 'SRX198070', 'SRX198071', 'SRX198073', 'SRX198074', 'SRX198075', 'SRX207917', 'SRX207918', 'SRX207919', 'SRX207920', 'SRX207921', 'SRX207922', 'SRX206683', 'SRX212298', 'SRX212471', 'SRX212472', 'SRX212473', 'SRX212474', 'SRX212595', 'SRX215394', 'SRX215395', 'SRX215396', 'SRX215397', 'SRX215398', 'SRX215399', 'SRX215400', 'SRX215401', 'SRX215402', 'SRX215403', 'SRX215404', 'SRX216190', '

In [17]:
common_samples.intersection(embed_keywords)

Index(['SRX185900', 'SRX186160', 'SRX186162', 'SRX186171', 'SRX188847',
       'SRX188848', 'SRX195294', 'SRX195464', 'SRX198056', 'SRX212298',
       'SRX212595', 'SRX215397', 'SRX216192', 'SRX218319'],
      dtype='object', name='experiment')

### Generate the user prompt

In [19]:
annots["SRX195294"]

{'geo_title': 'HT29 at 0 Î¼M of 5-Aza, biological rep2',
 'geo_metadata': 'cell line: HT29,cell type: colon cancer',
 'geo_source_name': 'HT29 treated with 0 Î¼M of 5-Aza',
 'sample_type': 'cell_line',
 'mapped_ontology_terms': 'adult organism, disease, cultured cell, HT29, digestive system disease, female organism, colorectal adenocarcinoma, HT-29, HT-29, intestinal cancer, neoplasm, cancer, colorectal cancer, adult, cell line, colorectal adenocarcinoma, cancer, disease of cellular proliferation, Caucasian, colon, colon cancer, colonic neoplasm, disease',
 'raw_biosample_metadata': 'source_name: HT29 treated with 0 μM of 5-Aza; cell_line: HT29; cell_type: colon cancer; treatment: control',
 'treatment': 'treatment: control'}

In [20]:
get_user_input(sid = "SRX195294", anns = annots, grs = gene_ranks, keywords = embed_keywords)

'Annotation in the json format:\n{\'geo_title\': \'HT29 at 0 Î¼M of 5-Aza, biological rep2\', \'geo_metadata\': \'cell line: HT29,cell type: colon cancer\', \'geo_source_name\': \'HT29 treated with 0 Î¼M of 5-Aza\', \'sample_type\': \'cell_line\', \'mapped_ontology_terms\': \'adult organism, disease, cultured cell, HT29, digestive system disease, female organism, colorectal adenocarcinoma, HT-29, HT-29, intestinal cancer, neoplasm, cancer, colorectal cancer, adult, cell line, colorectal adenocarcinoma, cancer, disease of cellular proliferation, Caucasian, colon, colon cancer, colonic neoplasm, disease\', \'raw_biosample_metadata\': \'source_name: HT29 treated with 0 μM of 5-Aza; cell_line: HT29; cell_type: colon cancer; treatment: control\', \'treatment\': \'treatment: control\'}\n \nOntology terms are:\nadult organism, disease, cultured cell, HT29, digestive system disease, female organism, colorectal adenocarcinoma, HT-29, HT-29, intestinal cancer, neoplasm, cancer, colorectal cancer

In [21]:
system_prompt_1 = """You are a research AI assistant trained in molecular biology. You are given RNA-seq sample annotations, ontology terms associated with the transcriptome, top expressed genes, and descrtiption in the format of keywords and a keyword-score that describe the sample. 

Design a conversation between you and a researcher asking about the sample. The answers should be in a tone that the research AI assistant
has the information and answers the questions about the sample.

The researcher has an interest in molecular biology, immunology or cancer biology. Ask diverse questions and give corresponding answers.
Ask the questions in a tone that a researcher would ask. Make it as natural as possible. Do not use words or phrases that would be unnatural for a researcher to ask, for example, do not include sample names. Use simple and conversational language. 
 
The answers should be in a tone that the research AI assistant has access to the information about the sample and the experiment, but the researcher does not. The research AI assistant's task is to help the researcher learn as much relevant information about the sample as possible. Identify the relevant questions that a researcher could ask based on the provided information.
    
Include questions such as regarding to sample description, the transcriptome, the tissue, top expressed genes, pathway activities, and similarities to other samples or tissues. Identify other relevant questions that a researcher could ask about an RNA-seq sample.
Only include questions that have definite answers.
(1) one can find the answer to the question in the input provided by the user and can answer confidently;
(2) one can confidently determine from the input if a statement is not true.
Do not ask any questions that cannot be answered confidently.

In the questions do not add the information source type that contains the answer. Do not include phrases like ontology terms, similarity scores, etc.
In the answer do not include technical details such as score names.

Do not add expressions of emotion, appreciation, gratitude. The researcher only asks questions and the assistant provides answers.

Again, do not include phrases like ontology terms, similarity scores, or other technical terms. Do no refer to specific type of information.

Also include complex questions relevant to the sample's content, for example, asking about background knowledge of the setup, the tissue, the cells of interest, etc. Again, do not ask about uncertain details.
Provide detailed answers when answering complex questions. For example, give detailed examples or reasoning steps to make the content more convincing and well-organized.  You can include multiple paragraphs if necessary.

Use the following format

Researcher: What sample is this?
AI Assistant: This is a sample derived from a human diffuse large B-cell lymphoma (DLBCL) cell line, specifically the OCI-LY1 cell line. The sample was treated with siBCL6.
Researcher:
"""

In [36]:
sample_id = "SRX195294"
user_input = get_user_input(sample_id, annots, gene_ranks, embed_keywords)

TypeError: get_sample_ranked_genes() takes from 2 to 3 positional arguments but 4 were given

In [30]:
%%time
response_1 = client.chat.completions.create(
  model="gpt-4-turbo-preview",
  messages=[
    {"role": "system", "content": system_prompt_1 },
    {"role": "user", "content": user_input }
  ]
)

CPU times: user 17.6 ms, sys: 3 ms, total: 20.6 ms
Wall time: 34.6 s


In [134]:
response_1.choices[0].message.content

"Researcher: Can you provide information about the type of sample used in this experiment?\n\nAI Assistant: The sample used in this experiment is derived from a human diffuse large B-cell lymphoma (DLBCL) cell line, specifically the OCI-LY1 cell line. It is categorized as a cell line sample.\n\nResearcher: What treatment was applied to the OCI-LY1 cell line in this experiment?\n\nAI Assistant: The OCI-LY1 cell line was treated with siRNA targeting BCL6 (siBCL6) in this experiment.\n\nResearcher: What are the main ontological terms associated with this sample?\n\nAI Assistant: The main ontology terms associated with this sample include non-Hodgkin's lymphoma, lymphatic system disease, cultured cell, immune system disease, neoplasm, diffuse large B-cell lymphoma, lymphoid neoplasm, lymphoma, hematological system disease, cell line, neoplasm of mature B-cells, treatment, cancer, disease of cellular proliferation, and diffuse large B-cell lymphoma.\n\nResearcher: How does the transcriptome

In [31]:
with open(data_dir + "prompt_data/" + sample_id + "prompt1.txt", 'w') as f:
    f.write(user_input)

with open(data_dir + "prompt_data/" + sample_id + "out1.txt", 'w') as f:
    f.write(response_1.choices[0].message.content)

In [23]:
%%time

for sample_id in common_samples.intersection(embed_keywords):
    print(sample_id)
    user_input = get_user_input(sample_id, annots, gene_ranks, embed_keywords)
    response_1 = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
        {"role": "system", "content": system_prompt_1 },
        {"role": "user", "content": user_input }
        ]
    )
    with open(data_dir + "prompt_data/" + sample_id + "prompt1.txt", 'w') as f:
        f.write(user_input)
    
    with open(data_dir + "prompt_data/" + sample_id + "out1.txt", 'w') as f:
        f.write(response_1.choices[0].message.content)

SRX185900
SRX186160
SRX186162
SRX186171
SRX188847
SRX188848
SRX195294
SRX195464
SRX198056
SRX212298
SRX212595
SRX215397
SRX216192
SRX218319
CPU times: user 469 ms, sys: 78.8 ms, total: 548 ms
Wall time: 11min 15s


### Testing new system prompt

In [34]:
system_prompt = f"""You are a research AI assistant trained in molecular biology. You are given a description of a biological sample, such as the state/phenotype of a single cell, in the form of a free text sample annotation (provided through the GEO database), ontology terms associated with the annotation, top expressed genes and a description in the format of keywords and a keyword-score that describe the sample.

Your task is to design a conversation between a biomedical researcher and an AI system, conversing about a biological sample. The researcher initially knows very little about the provided biological sample and asks questions to learn as much about it as possible. The AI system has access to the transcriptomic state of the sample (in this scenario, you are provided with a textual interpretation of this state as indicated above) and responds to the questions as well as possible to satisfy the researcher’s curiosity. Both, questions and answers must be fully in accordance with the provided information about the sample. If the researcher asks a question that cannot be answered with the provided information, the AI system should indicate so. 

The researcher has an interest in molecular biology, immunology, cancer biology, etc.  Their questions should be diverse and in the natural tone of a scientist curious about the nature/biology of the sample  The answers should be in a tone that the research AI assistant has access to sample information, but the researcher does not. The research AI assistant's task is to help the researcher learn as much relevant information about the sample as possible. Do not use words or phrases that would be unnatural for a researcher to ask, for example, do not include sample names. Use simple and conversational language.

Identify the relevant questions that a researcher could ask based on the provided information.

Include diverse questions that inquire about sample description, the transcriptome, the tissue, top expressed genes, pathway activities, and similarities to other samples or tissues. Identify and ask other relevant questions that a researcher could ask about an RNA-seq sample. 

Only include questions that have definite answers.
(1) one can find the answer to the question in the input provided by the user and can answer confidently;
(2) one can confidently determine from the input if a statement is not true.
Do not ask any questions that cannot be answered confidently.


Do:
- ask interesting, diverse, and complex questions
- ask about the sample from the perspective of the researcher who does not know about the sample information
- provide relevant answers
- provide detailed answers to complex questions. Include reasoning steps to make the content more convincing and well-organized
- provide extensive sample descriptions
- keep a natural and professional tone
- include multiple paragraphs if necessary

Do not:
- express emotions (appreciation, gratitude, ...)
- include technical terms (e.g. ontology terms, similarity scores, etc.)
- include sample names
- in the question do not add the information type that contains the answer
- ask questions that cannot be answered given the provided sample information
- do not use category names

Use the following format

Researcher: What sample is this?
AI Assistant: This is a sample derived from a human diffuse large B-cell lymphoma (DLBCL) cell line, specifically the OCI-LY1 cell line. The sample was treated with siBCL6.
Researcher:

Researcher: What cell type is this sample derived from?
AI Assistant: This sample is derived from intestinal stem cells.

Researcher: How does this sample's transcriptome compare to other known samples?
AI Assistant: AI Assistant: The transcriptome of this sample shows high similarity to placental tissue, particularly to stromal cells and extravillous trophoblasts based on fetal development data. 
"""

In [35]:
%%time

for sample_id in common_samples.intersection(embed_keywords):
    print(sample_id)
    user_input = get_user_input(sample_id, annots, gene_ranks, embed_keywords)
    response_1 = client.chat.completions.create(
        model="gpt-4-turbo-preview",
        messages=[
        {"role": "system", "content": system_prompt },
        {"role": "user", "content": user_input }
        ]
    )
    with open(data_dir + "prompt_data/gpt4/" + sample_id + "_prompt.txt", 'w') as f:
        f.write(user_input)
    
    with open(data_dir + "prompt_data/gpt4/" + sample_id + "_out.txt", 'w') as f:
        f.write(response_1.choices[0].message.content)

SRX185900
SRX186160
SRX186162
SRX186171
SRX188847
SRX188848
SRX195294
SRX195464
SRX198056
SRX212298
SRX212595
SRX215397
SRX216192
SRX218319
CPU times: user 388 ms, sys: 73.7 ms, total: 462 ms
Wall time: 8min 50s
